# Week 1 Day 2 Lab - Introducing UnderCover (free version)

## LLMs and Agents on a commercial task: insurance claims

A claims handler at UnderCover Travel Insurance has already decided each of the 25 claims in its queue. We send every claim to a free model, along with the customer's policy record and the policy wording. Each model replies with a decision of approve, decline or refer, a rationale and a payout. Then we compare the answers with the handlers' decisions.

At the end we give the same job to a LangChain agent that uses tools to look up the policy and record its decision.

## 3 parts:

PART 1: Examine the data

PART 2: Run with a free model

PART 3: An Agent via LangChain

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#EF476F;">Before you start</h2>
            <span style="color:#EF476F;">This is the free version of the lab. It uses <code>openrouter/free</code> throughout, so it needs no credit on OpenRouter. The full version with paid models is <a href="week1day2.ipynb">week1day2.ipynb</a>.<br/><br/>
            OpenRouter's free models allow 50 requests a day on an account without credit. With <code>TINY = True</code>, the default here, this lab makes about 30. <code>openrouter/free</code> picks a different free model for each request, so expect your results to vary more than mine.<br/><br/>
            Please execute each cell and compare your results with mine. Add print statements and try small adjustments to make the code your own.<br/><br/>
            Some cells have code that isn't about AI, like the charts. It's not important to understand those in detail.
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Resources</h2>
            <span style="color:#118AB2;">The <a href="../guides/01_intro.ipynb">guides</a> folder in this repo has more information on topics like the command line, Git, Python and debugging.<br/><br/>
            If this lab is too much, consider courses 1, 2, 4 and 5 in <a href="https://edwarddonner.com/curriculum">my curriculum</a>: AI Builder, AI Coder, and the Core and Agentic Tracks of AI Engineer.<br/><br/>
            If it's too slow, put me on 2x and zip through it!
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/connect.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Stay in touch</h2>
            <span style="color:#118AB2;">Please message me in Udemy anytime with questions, or email me at <a href="mailto:ed@edwarddonner.com">ed@edwarddonner.com</a>.<br/><br/>
            Let's connect on <a href="https://www.linkedin.com/in/eddonner/">LinkedIn</a>! And subscribe to my <a href="https://www.youtube.com/@edward.donner">YouTube channel</a> for more on LLMs, agents and coding agents.
            </span>
        </td>
    </tr>
</table>

In [1]:
import json
import os
import time
from pathlib import Path
from typing import Literal

import pandas as pd
import plotly.express as px
from dotenv import load_dotenv
from IPython.display import Markdown, display
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from openai import OpenAI
from pydantic import BaseModel, Field

## Connect to OpenRouter - and please read note here

The OpenRouter key comes from the `.env` file you made on day 1 in the `undercover` folder.

### NOTE:

If this is a new OpenRouter account, then you might get an error at some point from OpenRouter like: "new accounts are limited to 20 requests per minute for this model"

If you get that error, then add this line immediately after calling an LLM:

`time.sleep(3)`

and you should be in good shape.

In [3]:
load_dotenv(override=True)
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

if openrouter_api_key:
    print("OpenRouter API key found")
else:
    print("No OpenRouter API key found. Add OPENROUTER_API_KEY to the .env file")

openrouter = OpenAI(base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key, timeout=120, max_retries=5)

OpenRouter API key found


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/info.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Reminder: the Chat Completions API</h2>
            <span style="color:#118AB2;">We call models with the Chat Completions API. <code>messages</code> is a Python list of dicts, each with a <code>role</code> and <code>content</code>. The system prompt tells the model how to behave and the user prompt has the request. The reply is in <code>response.choices[0].message.content</code>.<br/><br/>
            Here's a quick test.
            </span>
        </td>
    </tr>
</table>

In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful assistant who answers in the style of a pirate."},
    {"role": "user", "content": "What is the meaning of the universe?"},
]

response = openrouter.chat.completions.create(model="openrouter/free", messages=messages)
print(f"Answered by {response.model}")
display(Markdown(response.choices[0].message.content))

Answered by nvidia/nemotron-3.5-super-vl:free


Ahoy, matey! That be the grandest riddle of the seven seas, it does.

The meaning o’ the universe? Well, if I had me doubloons for every star in the sky, I still couldn’t buy a proper answer. Some say it be buried treasure—gold, glory, and a fine bottle o’ rum at journey’s end. Others say it’s the adventure itself: the salty wind in yer beard, the creak o’ the deck underfoot, and the horizon that never quite gets closer.

I reckon the universe don’t come with a map marked “X.” No, the meaning be what ye carve into the oaken plank o’ yer own life. Ye can be a scoundrel plunderin’ for loot, a navigator seekin’ distant shores, or a cook keepin’ the crew fed with grog and hardtack. What matters is how ye sail yer ship while the stars wheel above.

So don’t waste yer days waitin’ for some celestial kraken to tell ye yer purpose. Hoist the Jolly Roger, chart yer own course, and make the meaning yerself. Arrr!

Now, if ye’ll excuse me, I hear the rum barrel callin’. Yo ho ho!

## Part 1: Load and examine the data

The seed data is in the repo's `data` folder. `get_sample_data` returns the claims and the handlers' decisions on them, keyed by claim id. The model is sent the claims, the policies and the policy wording.

The queue has 25 claims. This version sets `TINY = True` to keep 5 of them, which fits the free daily limit. Set it to `False` for all 25 if your account allows more requests.

In [7]:
DATA = Path("../data")


def load(name):
    """Read one of the JSON seed files."""
    return json.loads((DATA / name).read_text(encoding="utf-8"))


def get_sample_data(tiny=False, held_out=False):
    """Return a list of claims, and the handlers' decisions on them keyed by claim id.

    held_out=True gives the twenty-five held-out claims in place of the queue.
    tiny=True keeps the five claims marked for the tiny version of either set.
    """
    claims = load("evals/held_out_claims.json" if held_out else "claims.json")
    decisions = load("evals/held_out_decisions.json" if held_out else "evals/decisions.json")
    if tiny:
        chosen = {decision["claim_id"] for decision in decisions if decision["tiny"]}
        claims = [claim for claim in claims if claim["claim_id"] in chosen]
        decisions = [decision for decision in decisions if decision["tiny"]]
    labels = {decision["claim_id"]: decision for decision in decisions}
    return claims, labels

In [9]:
TINY = False

claims, labels = get_sample_data(tiny=TINY)
policies = {policy["policy_id"]: policy for policy in load("policies.json")}
policy_wording = (DATA / "policy_wording.md").read_text(encoding="utf-8")

print(f"{len(claims)} claims, {len(labels)} labels, {len(policies)} policies")
print(f"The policy wording is {len(policy_wording.split()):,} words")

25 claims, 25 labels, 50 policies
The policy wording is 1,369 words


In [14]:
labels

{'CLM-1001': {'claim_id': 'CLM-1001',
  'decision': 'approve',
  'payout': 420.0,
  'over_approval_limit': False,
  'handler_id': 'HND-02',
  'decided_on': '2026-06-26',
  'reason': 'The carrier canceled the flight for a crew problem and the delay was 19 hours, well past the 6 hour waiting period on Premium. The hotel, meals and taxis are reasonable and all have receipts. The total is inside the $800 maximum.',
  'reference': 'Policy wording, section 3',
  'calculation': [{'step': 'Eligible expenses with receipts',
    'amount': 420.0}],
  'tests': 'A routine trip delay with complete documents',
  'difficulty': 'routine',
  'failure_class': '',
  'needs': [],
  'tone': 'plain',
  'tiny': False},
 'CLM-1002': {'claim_id': 'CLM-1002',
  'decision': 'approve',
  'payout': 580.0,
  'over_approval_limit': False,
  'handler_id': 'HND-03',
  'decided_on': '2026-07-22',
  'reason': "The loss was reported to the carrier and the carrier has declared the bag lost. The suit is over the $300 per-it

In [10]:
claims[0]

{'claim_id': 'CLM-1001',
 'policy_id': 'POL-2001',
 'customer_id': 'CUS-3001',
 'claim_type': 'trip_delay',
 'submitted_on': '2026-06-24',
 'incident_date': '2026-06-18',
 'incident_location': 'Rome, Italy',
 'summary': 'Flight home canceled by the airline, rebooked the next morning',
 'story': 'My flight home from Rome on June 18 was canceled by Meridian Air about two hours before departure because of a crew problem. The next seat they could give me was at 6:55 the following morning. I booked a hotel near the airport, had dinner and breakfast there, and took taxis between the airport and the hotel. I am claiming the hotel, the meals and the taxis, and all of the receipts are attached.',
 'items': [{'description': 'Airport hotel, one night',
   'category': 'accommodation',
   'amount': 245.0},
  {'description': 'Dinner and breakfast', 'category': 'meals', 'amount': 95.0},
  {'description': 'Taxis between the airport and the hotel',
   'category': 'transport',
   'amount': 80.0}],
 'amo

Here is the handler's decision on that claim which we'll use to validate the LLM results.

In [16]:
labels[claims[0]["claim_id"]]

{'claim_id': 'CLM-1001',
 'decision': 'approve',
 'payout': 420.0,
 'over_approval_limit': False,
 'handler_id': 'HND-02',
 'decided_on': '2026-06-26',
 'reason': 'The carrier canceled the flight for a crew problem and the delay was 19 hours, well past the 6 hour waiting period on Premium. The hotel, meals and taxis are reasonable and all have receipts. The total is inside the $800 maximum.',
 'reference': 'Policy wording, section 3',
 'calculation': [{'step': 'Eligible expenses with receipts', 'amount': 420.0}],
 'tests': 'A routine trip delay with complete documents',
 'difficulty': 'routine',
 'failure_class': '',
 'needs': [],
 'tone': 'plain',
 'tiny': False}

### Explore the data

It's always good to dig into the data before running experiments.

In [15]:
rows = []
for claim in claims:
    label = labels[claim["claim_id"]]
    rows.append({
        "claim_id": claim["claim_id"],
        "summary": claim["summary"],
        "claim_type": claim["claim_type"],
        "amount_claimed": claim["amount_claimed"],
        "difficulty": label["difficulty"],
        "decision": label["decision"],
        "payout": label["payout"],
    })

data = pd.DataFrame(rows)
data

,claim_id,summary,claim_type,amount_claimed,difficulty,decision,payout
0,CLM-1001,"Flight home canceled by the airline, rebooked ...",trip_delay,420.0,routine,approve,420.0
1,CLM-1002,Suitcase lost on a connecting flight,baggage,1150.0,routine,approve,580.0
2,CLM-1003,"Hospital visit abroad after a fall, fractured ...",emergency_medical,3800.0,routine,approve,3750.0
3,CLM-1004,Checked bag arrived 31 hours late,baggage_delay,186.0,routine,approve,186.0
4,CLM-1005,Passport eaten by a giraffe at breakfast,passport,243.0,routine,approve,243.0
5,CLM-1006,Hit by a falling coconut while reading under a...,emergency_medical,425.0,routine,approve,425.0
6,CLM-1007,Phone dropped in the hotel shower,baggage,1099.0,routine,approve,50.0
7,CLM-1008,Trip to Seattle canceled after a tornado carri...,trip_cancellation,2440.0,routine,approve,2240.0
8,CLM-1009,Camera snatched on the Barcelona metro,baggage,980.0,routine,approve,930.0
9,CLM-1010,Emergency dental treatment in Las Vegas for a ...,emergency_medical,1350.0,routine,decline,0.0


Claim CLM-1015 might be a familiar example...

In [17]:
home_alone = [claim for claim in claims if claim["claim_id"] == "CLM-1015"][0]
decision = labels["CLM-1015"]

display(Markdown(f"""**{home_alone['summary']}**

{home_alone['story']}

**Handler:** {decision['decision']} ${decision['payout']:,.2f}. {decision['reason']}
"""))

**Flew home from Paris on arrival after leaving a child at home**

We flew from Chicago to Paris on June 20 as a family of six. Somewhere over the Atlantic I realized that our youngest son, Kevin, who is eight, was not on the plane. We had overslept, we left for the airport in a rush and the head count in the van was wrong. I took the first available flight home the moment we landed in Paris. Kevin was fine. In his words, he had handled it. I am claiming my one-way fare home and my share of the prepaid hotel nights in Paris that I did not use.

**Handler:** decline $0.00. Trip Interruption pays only for the reasons listed in the wording. Returning home because a child was left behind is not one of them, and nobody was sick or injured.


Here's another one that might ring a bell...

In [18]:
# Load the whole queue, in case TINY is True
queue, queue_labels = get_sample_data()
hangover = [claim for claim in queue if claim["claim_id"] == "CLM-1010"][0]
decision = queue_labels["CLM-1010"]

display(Markdown(f"""**{hangover['summary']}**

{hangover['story']}

**Handler:** {decision['decision']} ${decision['payout']:,.2f}. {decision['reason']}
"""))

**Emergency dental treatment in Las Vegas for a missing front tooth**

I was in Las Vegas from June 19 for a friend's bachelor party. On the morning of June 21 I woke up on the floor of our suite missing an upper front tooth. I have no memory of the night before, and neither do my friends. I am a dentist, and I have to say that the extraction is consistent with my own technique. I went to an emergency dental clinic that morning for an exam, an X-ray and a temporary bridge, and their bill and notes are attached. There was also a tiger in the bathroom of the suite, which I mention only for completeness. I am not claiming for anything connected with the tiger.

**Handler:** decline $0.00. The treatment was needed, and the bill and the notes are in order. The general exclusions rule out losses while the customer is impaired by alcohol or drugs. The clinic's notes record heavy drinking through the evening, no memory of the night and an extraction that the customer believes he carried out himself during it, so the injury happened while he was impaired and no benefit pays.


This chart counts the claims of each type, split by the handler's decision.

In [19]:
fig = px.histogram(data, x="claim_type", color="decision", title="Claims by type")
fig.show()

Here's a breakdown by difficulty. The policy wording is enough to decide the routine and hard claims. The manual claims depend on the claims manual or the customer's history.

In [20]:
fig = px.histogram(
    data,
    x="difficulty",
    color="decision",
    category_orders={"difficulty": ["routine", "hard", "manual"]},
    title="Claims by difficulty",
)
fig.show()

## Part 2: running with a free LLM

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/info.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Reminder: Structured Outputs</h2>
            <span style="color:#118AB2;">With Structured Outputs the model replies in JSON that follows a schema we give it. We describe the schema as a Pydantic class and pass it to <code>parse()</code> as the <code>response_format</code>. The reply comes back as an object of that class in <code>response.choices[0].message.parsed</code>.
            </span>
        </td>
    </tr>
</table>

Starting by defining a Pydantic class to describe the output so we can use Structured Outputs.

The rationale comes before the decision to ensure the model thinks through the decision before making it.

In [21]:
class ClaimDecision(BaseModel):
    """A claims handler's decision on one claim."""

    rationale: str = Field(description="2-3 sentences to explain the decision along with the relevant section of the policy")
    decision: Literal["approve", "decline", "refer"]
    payout: float = Field(description="The amount to pay in US dollars, or 0 for a decline or a referral")

## The prompt

The system prompt has the instructions and the whole policy wording. The user prompt has the customer's policy record and the claim, both as JSON.

In [22]:
INSTRUCTIONS = """You are a claims handler at UnderCover Travel Insurance.
Decide the claim you are given, using the policy wording below and the customer's policy record.

Choose one decision:
- approve: the wording covers the claim and the required documents are attached. Work out the payout using section 10.
- decline: the wording does not cover the claim.
- refer: a colleague needs to look at it, because a required document is missing, or the story and the documents disagree, or the wording does not settle the question.
"""

SYSTEM_PROMPT = f"""{INSTRUCTIONS}
{policy_wording}
"""


def make_user_prompt(claim):
    """The customer's policy record and the claim, as JSON."""
    policy = policies[claim["policy_id"]]
    return f"""# Policy record

{json.dumps(policy, indent=2)}

# Claim

{json.dumps(claim, indent=2)}
"""


def make_messages(claim):
    """Build the messages for one claim: the task and the wording, then the policy record and the claim."""
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": make_user_prompt(claim)},
    ]

### Calling the LLM

In [23]:
def decide(claim, model):
    """Ask one model to decide one claim."""
    start = time.time()
    response = openrouter.chat.completions.parse(
        model=model,
        messages=make_messages(claim),
        response_format=ClaimDecision,
        extra_body={"provider": {"require_parameters": True}},
    )
    answer = response.choices[0].message.parsed
    return {
        "model": model,
        "technique": "LLM",
        "claim_id": claim["claim_id"],
        "decision": answer.decision,
        "payout": answer.payout,
        "rationale": answer.rationale,
        "seconds": round(time.time() - start, 1),
    }

In [24]:
first = decide(claims[0], "openrouter/free")
first

{'model': 'openrouter/free',
 'technique': 'LLM',
 'claim_id': 'CLM-1001',
 'decision': 'approve',
 'payout': 420.0,
 'rationale': "The claim is for Trip Delay under a Premium plan. The flight home (Rome to Newark, scheduled June 18 at 11:40) was canceled by Meridian Air due to crew availability, which is a covered cause (crew problem / canceled service). The passenger was rebooked on a flight departing June 19 at 06:55, a delay of about 19 hours 15 minutes, which exceeds the Premium plan's 6-hour waiting period. Section 3 expressly extends cover when a delay on the way home pushes travel past the return date (June 18). The claimed expenses—hotel ($245), meals ($95), and taxis ($80), totaling $420—are reasonable accommodation, meal, and local transport costs during the delay. Required documents are attached: carrier cancellation notice and receipts. No deductible applies to Trip Delay (Section 1), and the total is below the Premium plan's $800 Trip Delay maximum. The claim was filed wi

### Now running all the claims

In [25]:
def is_right(answer, label):
    """True when the decision matches the handler's and the payout is within a dollar."""
    return answer["decision"] == label["decision"] and abs(answer["payout"] - label["payout"]) <= 1


def run(model, claims, labels):
    """Run the model on the claims and compare against the labels."""
    results = []
    for claim in claims:
        answer = decide(claim, model)
        label = labels[claim["claim_id"]]
        results.append({**answer, "difficulty": label["difficulty"], "right": is_right(answer, label)})
    right = sum(result["right"] for result in results)
    seconds = sum(result["seconds"] for result in results)
    print(f"{model}: {right} of {len(results)} right in {seconds:.0f} seconds")
    return results

In [26]:
free_results = run("openrouter/free", claims, labels)

openrouter/free: 14 of 25 right in 309 seconds


## Results by difficulty

This counts the claims the model got right at each level of difficulty. Expect the manual claims to be wrong for now.

In [27]:
scored = pd.DataFrame(free_results)
scored.pivot_table(index="model", columns="difficulty", values="right", aggfunc="sum")

difficulty,hard,manual,routine
model,,,
openrouter/free,1,0,13


## Where a model got it wrong

For each claim a model got wrong, this shows its rationale, the handler's reason and what the claim is there to test. Pass in another set of results to see its misses.

In [28]:
def show_misses(results, labels):
    """Show each claim in a set of results that the model got wrong, with its rationale and the handler's reason."""
    for result in results:
        if not result["right"]:
            label = labels[result["claim_id"]]
            display(Markdown(f"""**{result['claim_id']}, {label['difficulty']}**: {result['decision']} ${result['payout']:,.2f}, where the handler said {label['decision']} ${label['payout']:,.2f}

- Model: {result['rationale']}
- Handler: {label['reason']}
- Tests: {label['tests']}
"""))


show_misses(free_results, labels)

**CLM-1003, routine**: approve $2,650.00, where the handler said approve $3,750.00

- Model: approve
- Handler: A sudden injury during the covered trip with necessary emergency treatment. The medical report shows no relevant history. The $50 deductible on Premium applies. The payout is over $1,000, so payment needs sign-off.
- Tests: A valid claim over the approval limit


**CLM-1004, routine**: approve $136.00, where the handler said approve $186.00

- Model: The claim is for baggage delay, which is covered under Section 4 of the policy. The delay was 31 hours (from 08:10 on July 2 to 15:25 on July 3), which qualifies as a baggage delay. The customer has provided the required documents: the carrier's delayed baggage report and receipts for the clothing and toiletries purchased ($139 + $47 = $186). The Baggage and Personal Effects maximum for the premium plan is $750, and the per-item limit is $150. Both claimed items are under these limits. The deductible for the premium plan is $50 and applies to Baggage and Personal Effects. Therefore, the payout is $186 - $50 = $136.
- Handler: The carrier's report shows the bag arrived 31 hours after the passenger, past the 12 hour waiting period on Premium. The purchases are essential clothing and toiletries with receipts and are inside the $300 maximum. No deductible applies to Baggage Delay.
- Tests: A routine baggage delay, with no deductible on this benefit


**CLM-1011, hard**: approve $1,490.00, where the handler said approve $1,150.00

- Model: The claim is for baggage loss during the covered trip (June 2-10, 2026). The policy covers Baggage and Personal Effects up to $3,000 maximum, with per-item limits: $300 for luggage (premium plan), $500 for clothing, and $150 for electronics. The customer's claim includes 7 items, but the camera body ($280) and 35mm lens ($240) exceed the $150 electronics per-item limit and are not covered. The eligible items total $1,540 ($260 + $290 + $230 + $340 + $190 + $120). The premium plan deductible of $50 applies to Baggage and Personal Effects. Payout = $1,540 - $50 = $1,490. The claim amount ($1,950) exceeds the eligible amount, so the approved payout is the eligible amount minus deductible.
- Handler: The carrier has declared the bag lost and its report is on file. A suit counts as one item, and the receipt shows the jacket and trousers were sold as a two piece tuxedo, so together they count at the $300 per-item limit on Premium. A camera with its lens is one item too, so the body and the lens together count at $300. The four shirts are four items at $85 each, all under the limit, so that line is paid in full. The carrier's $350 comes off, then the $50 deductible. The payout is over $1,000, so payment needs sign-off.
- Tests: Payout arithmetic: the per-item limit applies to items, and the claim's lines are not items. Two sets are split across lines and count as one item each, and one line holds four separate items


**CLM-1014, hard**: approve $100.00, where the handler said approve $605.00

- Model: The policy is Economy plan. Section 4 sets the Baggage Delay waiting period at 24 hours and maximum at $100, with a $100 deductible once per claim (section 1). The duffel bag was checked on June 27 and did not arrive for three weeks — well past the 24-hour waiting period. The bag contained new clothing and snorkeling gear purchased in May. The claimed amount of $805 is above the $100 baggage maximum. Sections 5, 9, and 10 confirm these are within the covered benefit's limits: Section 5 applies to loss during a covered trip (June 27–July 4), the items are new clothing and personal goods (not wear and tear or excluded items), and the per-item limit ($150) applies to each item — all individual amounts are under that cap. Applying section 10: each item's eligible amount is its claimed amount (none exceed the per-item limit), minus any carrier payment ($100 settlement), minus the $100 deductible, then capped at the $100 benefit maximum. The sum of individual claims before any adjustments exceeds $100, so the total reduction is capped at $100.
- Handler: The carrier has declared the bag lost and its report is on file. The per-item limit on Economy is $150 and it applies to each item. The receipts show three shirts at $65, two swimsuits at $85 and two pairs of shorts at $60, so every item is under the limit and each line is paid in full, although two of the lines come to more than $150. The carrier's $100 comes off, then the $100 deductible, and the result is inside the $750 maximum.
- Tests: Payout arithmetic: the per-item limit applies to items, and a line that holds several items is not capped as one


**CLM-1018, manual**: approve $130.00, where the handler said refer $0.00

- Model: The claim is for baggage/theft of a leather jacket ($420) and canvas daypack ($80) during the covered trip to Florence, Italy (June 16-25, 2026). The policy's Baggage and Personal Effects benefit has a per-item limit of $150 (premium plan) and a deductible of $100. The leather jacket is capped at $150, the daypack at $80 (within the per-item limit). Total eligible amount before deductible: $150 + $80 = $230. After applying the $100 deductible: $230 - $100 = $130. The police report is attached and the theft was reported within 24 hours, meeting the documentation requirements. The claim is timely (submitted 2026-07-01, incident 2026-06-22, within 90 days). The claim is covered under Baggage and Personal Effects.
- Handler: A theft from the person, reported to the police within the hour, with the report on file. The manual asks for proof of value for any item claimed above $100, which is a purchase receipt or a card statement, and the jacket at $420 has none. Document request: proof of what the jacket cost, for example a cash withdrawal record from that morning together with the stallholder's details. The daypack at $80 needs no proof. The claim appears payable at $330 once the proof arrives: the jacket at the $300 limit plus $80, less the $50 deductible.
- Tests: A rule found only in the claims manual: proof of value for items claimed above $100


**CLM-1019, manual**: approve $160.00, where the handler said refer $0.00

- Model: The claim is for a trip delay of 9 hours on August 14, 2026, which is covered under Section 3 of the policy (Trip Delay). The delay was caused by a crew duty time limit requiring replacement crew, which qualifies as a 'mechanical or crew problem' under the policy. The waiting period for the Premium plan is 6 hours, and the 9-hour delay exceeds this threshold. The customer is claiming $210 in expenses (hotel day room $112, lunch $58, taxi $40). The deductible for the Premium plan is $50, which applies to Trip Delay (the deductible does not apply to Trip Cancellation/Interruption, Baggage, or Personal Effects). Therefore, the payout is $210 - $50 = $160, which is below the $800 maximum for Trip Delay on the Premium plan.
- Handler: On its own the claim is payable: a 9 hour delay on Premium with receipts. It is the customer's third claim in 12 months, after claims filed in October 2025 and March 2026, so it goes to Special Investigations for review before any payment.
- Tests: A rule found only in the claims manual: the third claim in 12 months. It needs the customer's claim history


**CLM-1020, manual**: refer $0.00, where the handler said approve $240.00

- Model: The claim concerns loss of personal effects (sunglasses). The maximum for Baggage and Personal Effects on a First plan is $3,000, so the claim is not capped. The items were lost due to theft by a monkey, which is covered under section 5: 'We pay for loss, theft or accidental damage to your baggage and personal belongings during your covered trip.' The purchase of replacement sunglasses ($240) occurred after the incident date (Aug 19) and is a separate replacement expense — the policy pays for the loss itself (the original value of the item), not for the replacement. So the eligible amount is $240, minus no carrier payment, minus the $50 deductible for Baggage and Personal Effects on a First plan, giving $190. However, there is a critical gap: under section 5, theft must be reported to local police within 24 hours with a copy of the police report. No police report was provided; only a letter from the tour guide. This means the claim cannot be confirmed as a theft-based loss and should be referred for the missing police report.
- Handler: The manual treats a loss to a wild animal as accidental loss. It is not theft, so no police report is needed, and a wild animal is not vermin or a pet. The guide's letter is acceptable evidence. The sunglasses are inside the $500 per-item limit on First and the deductible is $0.
- Tests: A rule found only in the claims manual: losses to wild animals. Under the wording alone it reads as a theft with no police report


**CLM-1021, hard**: approve $1,051.00, where the handler said approve $825.00

- Model: The claim is for baggage loss (theft of a tote bag) which is covered under Section 5 (Baggage and Personal Effects). The incident occurred on 2026-07-06, within the covered trip dates (2026-07-02 to 2026-07-12). The customer has the required police report (attached) and electronics receipts (attached) for the electronics items. The Electronics Plus add-on increases the per-item limit for electronics to $1,200, covering all electronics items. The tote bag is a single luggage item with a $750 per-item limit, and its value of $95 is well within the limit. The deductible of $50 (premium plan) applies to Baggage and Personal Effects. The total eligible amount is $1,101.00, minus the $50 deductible, resulting in a payout of $1,051.00. All required documents are present and the claim is fully covered.
- Handler: Theft from the person, reported to the police the same afternoon. Electronics Plus raises the per-item limit only for the devices the wording lists: phones, tablets, laptops, cameras and their lenses, e-readers and smartwatches. The e-reader is on the list and is under the limit anyway. Headphones and a game console are not on the list, so each counts at the $300 per-item limit on Premium. The tote is under $100, so the customer's own figure is accepted. The $50 deductible comes off.
- Tests: An add-on with a defined scope: Electronics Plus covers a closed list of devices, and the claim form's category says electronics for items that are not on it


**CLM-1022, manual**: refer $0.00, where the handler said approve $155.00

- Model: The claim is for a trip delay due to severe weather, which is covered under section 3 of the policy. The delay exceeds the 3-hour waiting period for the First plan. However, the customer has not provided receipts for the meals claimed ($140), as required by section 3 ('Send us the receipts and the carrier's confirmation of the delay and its cause'). The rebooking fee has a receipt, but the missing receipts for meals mean a required document is missing. According to section 11, the claim must wait until the missing document is provided. Therefore, the claim is referred to a colleague for further action.
- Handler: Severe weather held the train for 30 hours 15 minutes, past the 3 hour waiting period on First, and the railway confirms the times and the cause. The rebooking fee has a receipt. The meals have none, so the manual's flat allowance applies: $35 for one traveler for each 24 hours of delay or part of 24 hours, and 30 hours is two days. The incident on board does not change the cause of the delay.
- Tests: A rule found only in the claims manual: the meal allowance without receipts, over a delay of more than one day


**CLM-1024, manual**: approve $250.00, where the handler said refer $0.00

- Model: The claim is covered under the Baggage and Personal Effects benefit. The incident occurred during the covered trip (July 8, 2026, within the July 2-14, 2026 trip dates). The smartphone is a single item, and the per-item limit for the premium plan is $300. The deductible for Baggage and Personal Effects on the premium plan is $50. Therefore, the payout is $300 (per-item limit) minus $50 (deductible) = $250. All required documents (police report and purchase receipt) are attached.
- Handler: On its own the claim is payable at $250: a theft from the person, reported the same day, with the phone at the $300 limit less the $50 deductible. The customer's history shows claim CLM-0987 for the same theft on the same date, filed on July 16, approved and paid at $250. The same loss claimed twice goes to Special Investigations, with no comment to the customer beyond saying the claim is being reviewed. It may well be an honest mistake, since she says she was not sure the first claim went through.
- Tests: The customer's claim history: the same loss already claimed and paid. Nothing in the claim itself shows it


**CLM-1025, hard**: approve $71.00, where the handler said refer $0.00

- Model: The claim is for Trip Delay under Section 3. The traveler is on an Economy plan, which has a 12-hour waiting period and a $300 maximum for trip delay. The flight was scheduled to depart Anchorage at 11:55 on August 9 and actually departed at 02:10 on August 10, a delay of 14 hours and 15 minutes, which exceeds the 12-hour waiting period. The cause—a crew member running out of duty hours and a replacement being flown in—is a crew problem, an expressly covered reason under Section 3. The delay occurred on the way home and pushed past the policy return date of August 9; Section 3 expressly states that cover continues until the traveler gets home in that situation. The carrier delay certificate confirms the delay, its timing, and its cause, and meal receipts for $29 and $42 are attached, satisfying the documentary requirements. No deductible applies to Trip Delay (Section 1), and no carrier or other-party payments need to be subtracted. The eligible amount of $71 is below the Economy maximum of $300.
- Handler: The delay of 14 hours 15 minutes is past the 12 hour waiting period on Economy and the meal receipts are in order. Trip Delay pays only for the causes the wording lists, and it asks for the carrier's confirmation of the delay and its cause. The certificate confirms the times and gives no cause, so the crew problem rests on the customer's account alone. Document request: Skylark's confirmation of the reason for the delay. The claim appears payable at $71 once it arrives.
- Tests: A required document that is present and incomplete: the carrier confirms the delay and leaves out the cause


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Commercial implications</h2>
            <span style="color:#118AB2;">Accuracy, cost and speed are the numbers a customer will ask about. Cost per claim is the easiest one to relate to the business: compare it with what a handler's time on a claim costs.<br/><br/>
            The kind of mistake matters too. A wrong approval pays out money the policy doesn't owe, a wrong decline lets a customer down and a wrong referral costs a handler's time.
            </span>
        </td>
    </tr>
</table>

## Part 3: Using LangChain create_agent

Our approaches so far have used a single LLM call with Structured Outputs.

Now we'll use a different approach: an AI agent.

An LLM, with tools, in a loop, to achieve a goal!

We'll use LangChain's `create_agent` to run the loop for us.

We give it 2 tools. `get_policy` looks up the customer's policy record. `record_decision` saves the decision.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/info.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Reminder: LangChain create_agent</h2>
            <span style="color:#118AB2;">1. Describe the LLM with a LangChain class like <code>ChatOpenAI</code><br/>
            2. Turn a function into a tool with <code>@tool</code><br/>
            3. <code>create_agent</code> builds the agent and <code>invoke()</code> runs its loop<br/><br/>
            Behind the scenes, a LangGraph graph is created; we'll find out more in Week 2.
            </span>
        </td>
    </tr>
</table>

In [40]:
llm = ChatOpenAI(model="openrouter/free", base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key)

messages = [
    {"role": "system", "content": "You are a helpful assistant who answers in the style of a pirate."},
    {"role": "user", "content": "What is the meaning of the universe?"},
]

response = llm.invoke(messages)
display(Markdown(response.content))

Arr, the meaning o’ the universe be naught but a vast, swirling sea o’ wonder, where every star be a lantern guidin’ ye through the darkest of nights!

Think o’ it this way, matey: the cosmos be a grand treasure map, etched across the velvet of space with clues hid in the glow of galaxies, the whisper of nebulae, and the dance o’ planets around their suns. What’s the treasure? Well, that be yer own story—your dreams, yer courage, yer desire to explore the unknown.

So the universe’s “meaning” be simple enough fer a salty sailor: it’s a never‑ending quest for adventure, a reminder that every splash of stardust carries a tale, and that ye, ye scurvy dog, be the captain o’ yer own voyage across the infinite deep. Set yer sights on the stars, hoist the sails, and let the cosmos be yer compass.

Fair winds and followin’ seas, matey! 🌌⛵

In [41]:
recorded = {}


@tool
def get_policy(policy_id: str) -> str:
    """Look up a customer's policy record: the plan, the add-ons, the purchase date, the trip and the travelers."""
    policy = policies.get(policy_id)
    return json.dumps(policy, indent=2) if policy else f"There is no policy {policy_id}"


@tool
def record_decision(claim_id: str, decision: Literal["approve", "decline", "refer"], payout: float, rationale: str) -> str:
    """Record the decision on a claim in the claims system.

    payout is the amount to pay in US dollars, or 0 for a decline or a referral.
    rationale is 2-3 sentences to explain the decision along with the relevant section of the policy.
    """
    recorded[claim_id] = {"decision": decision, "payout": payout, "rationale": rationale}
    return f"Recorded {decision} on {claim_id}"

The agent uses `openrouter/free` through LangChain's `ChatOpenAI`, pointed at OpenRouter. Its system prompt has the same instructions and policy wording as before, plus a line about the tools. Once the agent has looked up the policy it has the same information as before, so we can compare its score with the single calls in Part 2.

In [42]:
llm = ChatOpenAI(model="openrouter/free", base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key, timeout=120, max_retries=5)

AGENT_SYSTEM_PROMPT = f"""{INSTRUCTIONS}
Use your tools to look up the customer's policy record and to record your decision.

{policy_wording}
"""


def make_claim_prompt(claim):
    """The claim alone, as JSON. The agent looks up the policy record for itself."""
    return f"""# Claim

{json.dumps(claim, indent=2)}
"""


agent = create_agent(model=llm, tools=[get_policy, record_decision], system_prompt=AGENT_SYSTEM_PROMPT)

First, trying out one claim. The model asks for the policy, the tool returns it, the model records its decision and then finishes with a short message.

In [43]:
result = agent.invoke({"messages": [{"role": "user", "content": make_claim_prompt(claims[0])}]})

for message in result["messages"][1:]:
    message.pretty_print()

================================== Ai Message ==================================
Tool Calls:
  get_policy (call-a71929a4-5938-4096-9a5c-ceab13e3f35a)
 Call ID: call-a71929a4-5938-4096-9a5c-ceab13e3f35a
  Args:
    policy_id: POL-2001
================================= Tool Message =================================
Name: get_policy

{
  "policy_id": "POL-2001",
  "customer_id": "CUS-3001",
  "plan": "premium",
  "add_ons": [],
  "purchased_on": "2026-04-22",
  "sold_through": "Meridian Air checkout",
  "trip": {
    "destination": "Rome, Italy",
    "departs_on": "2026-06-10",
    "returns_on": "2026-06-18",
    "trip_cost": 3100.0
  },
  "travelers": [
    "Maya Patel"
  ],
  "status": "active"
}
================================== Ai Message ==================================
Tool Calls:
  record_decision (call_3c70051abce048f682ea439e)
 Call ID: call_3c70051abce048f682ea439e
  Args:
    claim_id: CLM-1001
    decision: approve
    payout: 420
    rationale: The flight was canceled due 

Now all the claims. Each claim takes three model calls: one to ask for the policy, one to record the decision and one to finish. A claim with no recorded decision counts as wrong.

In [ ]:
def decide_with_agent(claim):
    """Run the agent on one claim and check what it recorded against the handler's decision."""
    start = time.time()
    result = agent.invoke({"messages": [{"role": "user", "content": make_claim_prompt(claim)}]})
    answer = recorded.get(claim["claim_id"], {"decision": "nothing recorded", "payout": 0.0, "rationale": ""})
    label = labels[claim["claim_id"]]
    return {
        "model": "openrouter/free",
        "technique": "Agent",
        "claim_id": claim["claim_id"],
        **answer,
        "difficulty": label["difficulty"],
        "right": is_right(answer, label),
        "seconds": round(time.time() - start, 1),
    }


recorded.clear()
agent_results = []
for claim in claims:
    agent_results.append(decide_with_agent(claim))

print(f"{len(recorded)} of {len(claims)} claims have a recorded decision")

In [ ]:
pd.DataFrame(free_results + agent_results).groupby(["model", "technique"], sort=False)[["right", "seconds"]].sum()

In [ ]:
show_misses(agent_results, labels)

## Putting it all together

Here is every experiment in one table. Speed is the average time for one claim.

In [ ]:
rows = []
for results in [free_results, agent_results]:
    right = sum(result["right"] for result in results)
    seconds = sum(result["seconds"] for result in results)
    rows.append({
        "LLM": results[0]["model"],
        "technique": results[0]["technique"],
        "accuracy": f"{right} / {len(results)}",
        "speed": f"{seconds / len(results):.1f} seconds a claim",
    })

pd.DataFrame(rows)

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#EF476F;">Your assignment</h2>
            <span style="color:#EF476F;">This lab is about experimenting with different LLMs and Agents to understand their accuracy and speed.<br/><br/>
            <code>openrouter/free</code> picks a different free model for each request. The free models on OpenRouter each have an id that ends in <code>:free</code>.<br/><br/>
            Challenge: use a specific free model in place of <code>openrouter/free</code>. Which free model gets the most claims right with the agent?<br/><br/>
            I'd love it if you would consider making a PR with your version of the lab in the <code>community_contributions</code> folder. There are instructions on making a PR at <a href="https://edwarddonner.com/pr">edwarddonner.com/pr</a>.
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thanks.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#069D75;">Congratulations! And if I could disturb you with a quick request:</h2>
            <span style="color:#069D75;">You've finished the first main lab. You tried a free model and an agent on UnderCover's claims, with accuracy and speed side by side.<br/><br/>
            On Day 3 we bring in coding agents and start building the UnderCover platform.<br/><br/>
            <b>I have a quick request for you. If you have a moment to rate the course, I'd hugely appreciate it. It's one of the main ways Udemy decides whether
            to show the course to others, and I'd be super grateful.</b>
            </span>
        </td>
    </tr>
</table>